# Getting Started with SynthBench

This notebook walks you through running your first synthetic survey benchmark,
inspecting the results, and comparing models. By the end you'll understand what
SynthBench measures and how to interpret the scores.

**Target audience**: Data scientists who are skeptical of synthetic surveys and
want to verify the claims hands-on.

**No API key needed** for the first few cells — we use a random baseline provider.

## 1. Install SynthBench

Install the package from PyPI (or from the local repo if you've cloned it).

In [ ]:
# Install from PyPI
!pip install synthbench

# Or install from a local clone:
# !pip install -e /path/to/synthbench

## 2. Run a Smoke Benchmark (No API Key Needed)

The `random` provider generates uniformly random answers — no API key required.
The `smoke` suite uses 28 questions so this finishes in seconds.

This is the absolute floor: any real model should score higher than random.

In [ ]:
!synthbench run --provider random --suite smoke --samples 10 --output ./my_results

## 3. Load and Inspect the JSON Results

Every benchmark run produces a JSON file with the full results. Let's load it
and look at the structure.

In [ ]:
import json
from pathlib import Path

# Find the most recent result file
result_files = sorted(Path("./my_results").glob("*.json"))
assert result_files, "No result files found — did the benchmark run above succeed?"

with open(result_files[-1]) as f:
    result = json.load(f)

# Top-level structure
print("Top-level keys:", list(result.keys()))
print()

# Scores
print("Scores:")
for metric, value in result["scores"].items():
    print(f"  {metric}: {value:.4f}")
print()

# Peek at one question
q = result["per_question"][0]
print(f"Example question: {q['text']}")
print(f"  Options: {q['options']}")
print(f"  Human distribution: {q['human_distribution']}")
print(f"  Model distribution: {q['model_distribution']}")
print(f"  JSD: {q['jsd']:.4f}, Kendall tau: {q['kendall_tau']:.4f}")

## 4. Display the Score Card

SynthBench can regenerate a readable report from any saved JSON.

In [ ]:
# Display the score card for the most recent result
!synthbench report {str(result_files[-1])}

## 5. Compare Two Results

Let's run a second benchmark with the majority-baseline provider (always picks
the most popular answer) and compare. This shows you the statistical
significance testing built into SynthBench.

In [ ]:
# Run majority-baseline on the same suite
!synthbench run --provider majority-baseline --suite smoke --samples 10 --output ./my_results

# Now compare the two
result_files = sorted(Path("./my_results").glob("*.json"))
print(f"\nFound {len(result_files)} result files. Comparing...\n")
!synthbench compare {str(result_files[0])} {str(result_files[-1])}

## 6. Run with a Real API Key (OpenRouter)

Now let's benchmark an actual model. OpenRouter gives you access to many models
with a single API key. Set `OPENROUTER_API_KEY` in your environment.

**This cell costs real money** (a few cents for the smoke suite).

In [ ]:
import os

# Set your API key (or export OPENROUTER_API_KEY in your shell)
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."

if os.environ.get("OPENROUTER_API_KEY"):
    !synthbench run \
        --provider openrouter \
        --model anthropic/claude-haiku-4-5-20251001 \
        --suite smoke \
        --samples 10 \
        --output ./my_results
else:
    print("Skipped: set OPENROUTER_API_KEY to run this cell.")
    print('Example: os.environ["OPENROUTER_API_KEY"] = "sk-or-..."')

## 7. Topic Analysis — Consumer vs Political

SynthBench lets you slice by topic to see where a model excels or struggles.
Political questions are often harder than consumer-preference questions.

In [ ]:
# Run the random baseline on consumer and political subsets separately
!synthbench run --provider random --topic consumer --samples 10 --output ./my_results
!synthbench run --provider random --topic political --samples 10 --output ./my_results

# Compare scores across topics
topic_files = [
    f for f in sorted(Path("./my_results").glob("*.json")) if "random" in f.name
]
for f in topic_files[-2:]:
    with open(f) as fh:
        data = json.load(fh)
    topic = data.get("config", {}).get("topic", "overall")
    sps = data.get("scores", {}).get("sps", 0)
    print(f"  {topic}: SPS = {sps:.4f}")

## 8. Plug In Your Own Provider (HttpProvider)

Have your own survey-response API? SynthBench can benchmark any REST endpoint.
Your endpoint receives a POST with:

```json
{"question": "...", "options": ["A", "B", "C"], "persona": null}
```

And returns:

```json
{"selected_option": "B"}
```

Point SynthBench at it with `--provider http --url <your-endpoint>`.

In [ ]:
# Example: benchmark a custom endpoint
# Replace the URL with your actual endpoint

CUSTOM_URL = None  # e.g., "https://my-api.example.com/survey"

if CUSTOM_URL:
    !synthbench run \
        --provider http \
        --url {CUSTOM_URL} \
        --suite smoke \
        --samples 10 \
        --output ./my_results
else:
    print("Set CUSTOM_URL above to benchmark your own endpoint.")
    print()
    print("Your endpoint should accept POST with JSON body:")
    print('  {"question": "...", "options": ["A", "B"], "persona": null}')
    print("And return:")
    print('  {"selected_option": "A"}')